<a href="https://colab.research.google.com/github/hbetabessi/prompt_engineering_techniques/blob/main/basic_prompt_enigneering_nocrash.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q -U langchain-core langchain-groq
print("✅ Dependencies installed!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.4/542.4 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 9.0 MB/s eta 0:00:00
✅ Dependencies installed!


In [2]:
import os
import re
import getpass
from typing import List, Dict, Tuple
from dataclasses import dataclass
from IPython.display import display, Markdown

# Fixed Imports
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate

# Setup API Key

In [3]:
if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass.getpass("Enter Groq API Key: ")
print("✅ Setup complete!")

Enter Groq API Key: ··········
✅ Setup complete!


In [4]:
@dataclass
class SimplePrompt:
    """Simple prompt container"""
    text: str
    score: float = 0.0
    technique: str = ""
    temperature: float = 0.5

    def show(self):
        """Display the prompt nicely"""
        display(Markdown(f"""
        **Technique:** {self.technique} | **Score:** {self.score:.1f}/10 | **Temperature:** {self.temperature}

        ```
        {self.text}
        ```
        """))

In [5]:
class APE:
    """Automatic Prompt Engineer"""

    def __init__(self):
        self.llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0.5)
        print("APE ready!")

    def generate_prompts(self, task: str) -> List[SimplePrompt]:
        """Generate different types of optimized prompts"""
        print("🔄 Generating optimized prompts...")

        generator = ChatPromptTemplate.from_messages([
            ("system", """You are an expert prompt engineer. Create 5 different high-quality prompts for the given task using these techniques:
            1. ZERO_SHOT, 2. EXPERT_PERSONA, 3. STEP_BY_STEP, 4. EXAMPLE_BASED, 5. COMPREHENSIVE.
            Format exactly as: TECHNIQUE_NAME: [prompt text]"""),
            ("human", "Task: {task}")
        ])

        result = self.llm.invoke(generator.format_prompt(task=task))
        return self._parse_prompts(result.content)

In [7]:
class APE:
    """Automatic Prompt Engineer"""

    def __init__(self):
        self.llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0.5)
        print("APE ready!")

    def generate_prompts(self, task: str) -> List[SimplePrompt]:
        """Generate different types of optimized prompts"""
        print("🔄 Generating optimized prompts...")

        generator = ChatPromptTemplate.from_messages([
            ("system", """You are an expert prompt engineer. Create 5 different high-quality prompts for the given task using these techniques:
            1. ZERO_SHOT, 2. EXPERT_PERSONA, 3. STEP_BY_STEP, 4. EXAMPLE_BASED, 5. COMPREHENSIVE.
            Format exactly as: TECHNIQUE_NAME: [prompt text]"""),
            ("human", "Task: {task}")
        ])

        result = self.llm.invoke(generator.format_prompt(task=task))
        return self._parse_prompts(result.content)

    def _parse_prompts(self, response: str) -> List[SimplePrompt]:
        prompts = []
        techniques = ["ZERO_SHOT", "EXPERT_PERSONA", "STEP_BY_STEP", "EXAMPLE_BASED", "COMPREHENSIVE"]
        temperatures = [0.3, 0.4, 0.2, 0.5, 0.3]

        for i, technique in enumerate(techniques):
            pattern = f"{technique}:\s*(.*?)(?={techniques[i+1] if i+1 < len(techniques) else '$'})"
            match = re.search(pattern, response, re.DOTALL | re.IGNORECASE)
            if match:
                prompts.append(SimplePrompt(
                    text=match.group(1).strip(),
                    technique=technique.lower().replace('_', '-'),
                    temperature=temperatures[i]
                ))
        print(f"✅ Generated {len(prompts)} prompts")
        return prompts

    def evaluate_prompts(self, task: str, prompts: List[SimplePrompt]) -> List[SimplePrompt]:
        print("📊 Evaluating prompts...")
        evaluator = ChatPromptTemplate.from_messages([
            ("system", "Score prompts 1-10. Response format: Score: [number]/10"),
            ("human", "Task: {task}\nTechnique: {technique}\nPrompt: {prompt_text}")
        ])

        for prompt in prompts:
            result = self.llm.invoke(evaluator.format_prompt(task=task, technique=prompt.technique, prompt_text=prompt.text))
            score_match = re.search(r"Score:\s*(\d+(?:\.\d+)?)/10", result.content)
            prompt.score = float(score_match.group(1)) if score_match else 5.0

        return sorted(prompts, key=lambda x: x.score, reverse=True)

    def test_best_prompt(self, task: str, best_prompt: SimplePrompt) -> str:
        test_llm = ChatGroq(model="llama-3.1-8b-instant", temperature=best_prompt.temperature)
        tester = ChatPromptTemplate.from_messages([
            ("system", "Follow the prompt instructions carefully."),
            ("human", "{prompt}\n\nTask: {task}")
        ])
        return test_llm.invoke(tester.format_prompt(prompt=best_prompt.text, task=task)).content

    def run_ape(self, task: str):
        prompts = self.generate_prompts(task)
        evaluated_prompts = self.evaluate_prompts(task, prompts)
        best_prompt = evaluated_prompts[0]
        result = self.test_best_prompt(task, best_prompt)

        display(Markdown(f"### 🏆 Best Prompt Found: {best_prompt.technique}\nScore: {best_prompt.score}\n\n**Result:**\n{result}"))
        return best_prompt, result

<>:28: SyntaxWarning: invalid escape sequence '\s'
<>:28: SyntaxWarning: invalid escape sequence '\s'
/tmp/ipykernel_2699/2384920089.py:28: SyntaxWarning: invalid escape sequence '\s'
  pattern = f"{technique}:\s*(.*?)(?={techniques[i+1] if i+1 < len(techniques) else '$'})"


# TEST IT

In [8]:
ape = APE()
ape.run_ape("Explain how a battery works to a 10 year old")

APE ready!
🔄 Generating optimized prompts...
✅ Generated 5 prompts
📊 Evaluating prompts...


### 🏆 Best Prompt Found: comprehensive
Score: 8.0

**Result:**
Let's talk about batteries. You know how we use batteries to power our toys, flashlights, and other devices? Well, I'm going to explain how they work in a way that's easy to understand.

**The Inner Workings of a Battery**

Imagine a battery as a tiny factory that produces electricity. Inside the battery, there are three main parts:

1. **Positive Terminal (Plus Sign)**: This is the top part of the battery where you put the positive end of a wire or a device.
2. **Negative Terminal (Minus Sign)**: This is the bottom part of the battery where you put the negative end of a wire or a device.
3. **Electrolyte**: This is a special liquid or gel that helps the battery work. It's like a special kind of juice that makes the battery produce electricity.

**The Chemical Reaction**

When you connect a device to the battery, a chemical reaction starts to happen. Here's what happens:

* The positive terminal (plus sign) has tiny particles called electrons that want to escape.
* The negative terminal (minus sign) has a special kind of "vacancy" that needs electrons to fill it.
* The electrolyte helps the electrons from the positive terminal move to the negative terminal.
* As the electrons move, they create a flow of electricity that powers the device.

**Powering Everyday Objects**

So, how does this process translate to powering everyday objects? Here's an example:

* Let's say you have a flashlight that uses a battery to power a light bulb.
* When you connect the flashlight to the battery, the chemical reaction starts to happen, and electrons begin to flow from the positive terminal to the negative terminal.
* The electrons create a flow of electricity that powers the light bulb, making it shine bright!
* The same process happens with other devices like toys, radios, and even your phone. The battery produces electricity, which powers the device, allowing it to work.

In summary, batteries work by using a chemical reaction to produce electricity, which is then used to power devices. The positive and negative terminals, electrolyte, and electrons all work together to make this happen.

(SimplePrompt(text="Describe the inner workings of a battery, including the positive and negative terminals, the electrolyte, and the chemical reaction that occurs when it's connected to a device. Then, explain how this process translates to powering everyday objects.", score=8.0, technique='comprehensive', temperature=0.3),
 'Let\'s talk about batteries. You know how we use batteries to power our toys, flashlights, and other devices? Well, I\'m going to explain how they work in a way that\'s easy to understand.\n\n**The Inner Workings of a Battery**\n\nImagine a battery as a tiny factory that produces electricity. Inside the battery, there are three main parts:\n\n1. **Positive Terminal (Plus Sign)**: This is the top part of the battery where you put the positive end of a wire or a device.\n2. **Negative Terminal (Minus Sign)**: This is the bottom part of the battery where you put the negative end of a wire or a device.\n3. **Electrolyte**: This is a special liquid or gel that helps t